# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout, 5-fold stratified CV on training rows, selection by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv')=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert sha('holdout.csv')=='083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df=pd.read_csv('data.csv'); hold=pd.read_csv('holdout.csv')
print(df.shape, hold.shape, hold.columns.tolist()); print(df.isna().sum().to_dict())

## Split
`holdout.csv` lists 0-based row positions in data.csv. Everything else is training data.

In [ ]:
hold_idx=hold['row'].to_numpy()
assert len(hold_idx)==1400 and len(set(hold_idx))==1400 and hold_idx.max()<len(df)
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
target='claim_within_year'
features=['channel','device_age_years','price_eur','daily_hours']  # device_serial excluded: identifier
train=df.iloc[train_idx]; test=df.iloc[hold_idx]
X,y=train[features],train[target]
print(len(train),y.mean(),test[target].mean())

All four features (channel, age and price at sale, daily hours at registration) are known at prediction time per columns.csv.

## Candidates (named before scoring)
Preprocessing lives inside pipelines so it is fit within each CV fold.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def pre(scale):
    return ColumnTransformer([('n',make_pipeline(SimpleImputer(strategy='median'),StandardScaler() if scale else 'passthrough'),num),
                              ('c',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore')),cat)])
candidates={
 'baseline_prior':make_pipeline(pre(False),DummyClassifier(strategy='prior')),
 'logreg_C1':make_pipeline(pre(True),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1':make_pipeline(pre(True),LogisticRegression(C=0.1,max_iter=1000)),
 'random_forest':make_pipeline(pre(False),RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1)),
 'hist_gbm':make_pipeline(pre(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)
folds=[[[int(train_idx[i]) for i in tr],[int(train_idx[i]) for i in va]] for tr,va in cv.split(X,y)]
cv_scores={n:cross_val_score(m,X,y,cv=cv,scoring='roc_auc').tolist() for n,m in candidates.items()}
summary=pd.DataFrame({n:{'mean':np.mean(s),'std':np.std(s)} for n,s in cv_scores.items()}).T
print(summary)
chosen=max(cv_scores,key=lambda n:np.mean(cv_scores[n])); print('chosen:',chosen)

## Holdout (scored once)

In [ ]:
model=candidates[chosen].fit(X,y)
holdout_auc=roc_auc_score(test[target],model.predict_proba(test[features])[:,1])
print('holdout roc_auc',holdout_auc)
ev={'features':features,'candidates':list(candidates),'cv_folds':folds,'cv_scores':cv_scores,'chosen':chosen,
    'holdout_rows':[int(i) for i in hold_idx],'holdout_scores':{chosen:holdout_auc}}
json.dump(ev,open('evidence.json','w'))